In [24]:
from enum import Enum
from collections import deque
from dataclasses import dataclass, field, InitVar
from typing import Deque, List, Set, Dict, Tuple


class PacketType(Enum):    
    EXCESS = 0
    DEFICIT = 1
    BALANCED = 2
    UNDEFINED = 3


@dataclass
class EnergyPacket:
    capacity: float
    energy: float
        
    @property
    def capacity_max(self) -> float:
        return self.capacity + self.energy



@dataclass
class PhasePair:
    """ A phase pair consists of excess energy packets and deficit energy packets belonging to exaxtly one excess phase and one deficit phase.
    """
    energy_packets: Dict[PacketType, Deque[EnergyPacket]] = field(default_factory=lambda: {PacketType.EXCESS: deque(), PacketType.DEFICIT: deque()})
    n_unbalanced: Dict[PacketType, int] = field(default_factory = lambda: {PacketType.EXCESS: 0, PacketType.DEFICIT: 0})
    N_unbalanced_total: int = 0

    energy_excess_initial: InitVar[float | None] = None
    energy_deficit_initial: InitVar[float | None] = None
    
    def __post_init__(self, energy_excess_initial: float, energy_deficit_initial: float):
        self.insert_packet(
            index_packet=0,
            packet_type=PacketType.EXCESS,
            energy_packet=EnergyPacket(capacity=0, energy=energy_excess_initial)
        )
        self.insert_packet(
            index_packet=0,
            packet_type=PacketType.DEFICIT,
            energy_packet=EnergyPacket(capacity=0, energy=energy_deficit_initial)
        )
        
    
    @property
    def phase_type(self):
        if self.n_unbalanced[PacketType.EXCESS] == 0 and self.n_unbalanced[PacketType.DEFICIT] == 0:
            return PacketType.BALANCED
            
        if self.n_unbalanced[PacketType.EXCESS] >= 1 and self.n_unbalanced[PacketType.DEFICIT] >= 1:
            return PacketType.UNDEFINED

        if self.n_unbalanced[PacketType.EXCESS] >= 1:
            return PacketType.EXCESS

        if self.n_unbalanced[PacketType.DEFICIT] >= 1:
            return PacketType.DEFICIT

        raise


    def remove_packet(self, index_packet: int, packet_type: PacketType ):
        """Removes an energy packet of packet_type at index_packet from the phase pair.
        Rotate left by index_packet is O(index_packet), pop left O(1) and rotate right by index_packet O(index_packet)
        """
        self.energy_packets[packet_type].rotate(-index_packet)
        self.energy_packets[packet_type].popleft()
        self.energy_packets[packet_type].rotate( index_packet)

        self.n_unbalanced[packet_type] -= 1  # reduce the number of unbalanced packets
        self.N_unbalanced_total -= 1

    
    def insert_packet(self, index_packet: int, packet_type: PacketType, energy_packet: EnergyPacket ):
        """Inserts an energy packet of packet_type at index_packet to the phase pair.
        Rotate left by index_packet is O(index_packet), append left O(1) and rotate right by index_packet O(index_packet)
        """
        self.energy_packets[packet_type].rotate(-index_packet)
        self.energy_packets[packet_type].appendleft(energy_packet)
        self.energy_packets[packet_type].rotate( index_packet)
    
        self.n_unbalanced[packet_type] += 1  # increase the number of unbalanced packets
        self.N_unbalanced_total += 1

    
    def balance_packet(self):
        self.n_unbalanced[PacketType.EXCESS] -= 1
        self.n_unbalanced[PacketType.DEFICIT] -= 1
        self.N_unbalanced_total -= 2

        # rotate the packet-tuples left to move the now balanced packet to the end of the deque            
        self.energy_packets[PacketType.EXCESS].rotate(-1)
        self.energy_packets[PacketType.DEFICIT].rotate(-1)


    def merge_packets(self, packet_type: PacketType):
        while self.n_unbalanced[packet_type] > 1 and self.energy_packets[packet_type][0].capacity_max >= self.energy_packets[packet_type][1].capacity:
            print(f'Merging packets of type {packet_type}')
            print(f'Before: {self.energy_packets[packet_type]}')
            self.energy_packets[packet_type][0].energy += self.energy_packets[packet_type][1].energy  # combine the energy content
    
            # remove the merged packet
            self.remove_packet(
                index_packet=1,
                packet_type=packet_type
            )
            
            print(f'Now: {self.energy_packets[packet_type]}')

    
    def split_packet(self, index_packet: int, packet_type: PacketType, capacity_to_split: float):
        if capacity_to_split <= self.energy_packets[packet_type][index_packet].capacity or self.energy_packets[packet_type][index_packet].capacity_max <= capacity_to_split:
            return  # we can only split a packet, when the capacity value lies within the packet
    
        print(f'Splitting packet of type {packet_type} at {capacity_to_split}')
        print(f'Before: {self.energy_packets[packet_type]}')
        
        energy_new_packet = self.energy_packets[packet_type][index_packet].capacity_max - capacity_to_split  # calculate the remaining energy content
        self.energy_packets[packet_type][index_packet].energy = capacity_to_split - self.energy_packets[packet_type][index_packet].capacity  # calculate the lower packets energy content.
        
        # insert the new packet at index 1
        self.insert_packet(
            index_packet=index_packet+1,
            packet_type=packet_type,
            energy_packet=EnergyPacket(
                capacity=capacity_to_split, 
                energy=energy_new_packet
            )
        )
        
        print(f'Now: {self.energy_packets[packet_type]}')
    
    

@dataclass
class PhaseGroup:
    """
    A phase-group is collection of phase-pairs that can be compressed, balanced, and combined with other phase groups.
    A phase-group of type UNDEFINED will need to be balanced first and will then be either EXCESS, DEFICIT, or BALANCED.
    Phase-groups of type EXCESS or DEFICIT can be compressed by shifting the energy packets of the grouped phase pairs.
    """
    phase_pairs: Deque[PhasePair] = field(repr=False)
    
    group_type: PacketType
    index_start: int
    index_end: int = None
    
    index_target: int = None
    indices_to_shift: List[int] = field(default_factory=list)
    capacities_for_shift: List[int] = field(default_factory=list)


    def balance_group(self):
        """
        Balancing a group is only required for group_type==UNDEFINED and will only need to check the very first phase pair at index_start.
        """
        if self.group_type != PacketType.UNDEFINED:
            print(f'A group of type {self.group_type} cannot be balanced!')
            raise

        while self.phase_pairs[self.index_start].n_unbalanced[PacketType.EXCESS] > 0 and self.phase_pairs[self.index_start].n_unbalanced[PacketType.DEFICIT] > 0:            
            # The earliest available capacity is the maximum of both packets. Raise packets to the earliest available capacity which might touch the next energy packets of the same kind.
            capacity_earliest_available = max( self.phase_pairs[self.index_start].energy_packets[PacketType.EXCESS][0].capacity, self.phase_pairs[self.index_start].energy_packets[PacketType.DEFICIT][0].capacity )

            if self.phase_pairs[self.index_start].energy_packets[PacketType.EXCESS][0].capacity < capacity_earliest_available:
                self.phase_pairs[self.index_start].energy_packets[PacketType.EXCESS][0].capacity = capacity_earliest_available
                self.phase_pairs[self.index_start].merge_packets(PacketType.EXCESS)  # Merge and potentially remove packets.

            if self.phase_pairs[self.index_start].energy_packets[PacketType.DEFICIT][0].capacity < capacity_earliest_available:
                self.phase_pairs[self.index_start].energy_packets[PacketType.DEFICIT][0].capacity = capacity_earliest_available
                self.phase_pairs[self.index_start].merge_packets(PacketType.DEFICIT)  # Merge and potentially remove packets.

            # Balance excess and deficit by computing the minimum of both energy contents and create a new packet with the remaining content.
            capacity_max_excess = self.phase_pairs[self.index_start].energy_packets[PacketType.EXCESS][0].capacity_max
            capacity_max_deficit = self.phase_pairs[self.index_start].energy_packets[PacketType.DEFICIT][0].capacity_max
            capacity_to_split = min( capacity_max_excess, capacity_max_deficit )

            if capacity_max_excess > capacity_to_split:  # excess remaining
                # Split will create a new unbalanced energy packet
                self.phase_pairs[self.index_start].split_packet(index_packet=0, packet_type=PacketType.EXCESS, capacity_to_split=capacity_to_split)  
            elif capacity_max_deficit > capacity_to_split:  # deficit remaining
                # Split will create a new unbalanced energy packet
                self.phase_pairs[self.index_start].split_packet(index_packet=0, packet_type=PacketType.DEFICIT, capacity_to_split=capacity_to_split)  

            self.phase_pairs[self.index_start].balance_packet()

        
        self.group_type = self.phase_pairs[self.index_start].phase_type
        if self.group_type == PacketType.BALANCED:
            self.capacities_for_shift = [self.phase_pairs[self.index_start].energy_packets[PacketType.DEFICIT][-1].capacity_max]

    
    def merge_with(self, other: PhaseGroup):
        if self.group_type == PacketType.UNDEFINED or other.group_type == PacketType.UNDEFINED:
            raise ValueError("It is not possible to merge UNDEFINED groups, since they need to be subjected to the balance operation first to either be BALANCED, EXCESS, or DEFICIT.")

               
        
        match (self.group_type, other.group_type):
            case (PacketType.BALANCED, PacketType.BALANCED):
                """ Two groups of type BALANCED can be merged by updating the capacities_for_shift to the maximum of both"""
                self.capacities_for_shift[-1] = max(self.capacities_for_shift[-1], other.capacities_for_shift[-1])
            case (PacketType.BALANCED, PacketType.EXCESS):
                raise ValueError("It is not allowed to merge a BALANCED phase and an EXCESS phase, since we would loose information for potential later shift operations.")
            case (PacketType.BALANCED, PacketType.DEFICIT):
                raise ValueError("It is not allowed to merge a BALANCED phase and an DEFICIT phase, since we would loose information for potential later shift operations.")
            case (PacketType.EXCESS, PacketType.BALANCED):
                """An EXCESS group and a BALANCED phase can be merged by setting the last capacities_for_shift to the maximum of itself and the capacities_for_shift of the BALANCED group."""
                self.capacities_for_shift[-1] = max(self.capacities_for_shift[-1], other.capacities_for_shift[-1])
            case (PacketType.EXCESS, PacketType.EXCESS):
                """ Two groups of type EXCESS can be merged by appending the start index of the second one to indices_to_shift and the capacity of the first unbalanced packet at the start index to capacities_for_shift"""
                self.indices_for_shift.append(other.index_start)
                # append the capacity (lower end) of the fist unbalanced EXCESS at other.index_start
                self.capacities_for_shift.append(other.phase_pairs[other.index_start][PhaseType.EXCESS][0].capacity)
            case (PacketType.EXCESS, PacketType.DEFICIT):
                raise ValueError("It is not allowed merge an EXCESS phase and an DEFICIT phase. They need to shifted, which will create a new UNDEFINED group inplace of the DEFICIT group and remove the EXCESS one.")
            case (PacketType.DEFICIT, PacketType.BALANCED):
                """A DEFICIT group and a BALANCED phase can be merged by setting the last capacities_for_shift to the maximum of itself and the capacities_for_shift of the BALANCED group."""
            case (PacketType.DEFICIT, PacketType.EXCESS):
                raise ValueError("It is not allowed merge a BALANCED phase and an DEFICIT phase. They need to shifted, which will create a new UNDEFINED group inplace of the DEFICIT group and remove the EXCESS one.")
            case (PacketType.DEFICIT, PacketType.DEFICIT):
                """ Two groups of type DEFICIT can be merged by appending the start index of the second one to indices_to_shift and the capacity of the first unbalanced packet at the start index to capacities_for_shift"""

        
            """Merging two groups will allways set the end index of the first one to the end index of the second one."""
            self.index_end = other.index_end
            
            


@dataclass
class Context:
    energy_excess_per_phase_initial: List[float]
    energy_deficit_per_phase_initial: List[float]
    N_phases: int = 0
    
    phase_pairs: Deque[PhasePair] = None  # The algorithm will store results in this one
    
    phase_groups: Deque[PhaseGroup] = None  # The algorithm will work on this one

    indices_to_balance: Deque[int] = None
    indices_first_and_last: Dict[PacketType, List[int]] = field(default_factory=lambda: {PacketType.EXCESS: [None, None], PacketType.DEFICIT: [None, None]})
    
    @property
    def N_unbalanced_total(self):
        return sum([phase_pair.N_unbalanced_total for phase_pair in self.phase_pairs])

    @property
    def n_unbalanced_excess(self):
        return sum([phase_pair.n_unbalanced[PacketType.EXCESS] for phase_pair in self.phase_pairs])

    @property
    def n_unbalanced_deficit(self):
        return sum([phase_pair.n_unbalanced[PacketType.DEFICIT] for phase_pair in self.phase_pairs])

    
    def __post_init__(self):
        assert len(self.energy_excess_per_phase_initial) == len(self.energy_deficit_per_phase_initial)
        
        self.N_phases = len(self.energy_deficit_per_phase_initial)

        self.indices_to_balance = deque(range(self.N_phases))        

        self.phase_pairs = deque([PhasePair(
            energy_excess_initial=energy_excess_initial,
            energy_deficit_initial=energy_deficit_initial,
        ) for (energy_excess_initial, energy_deficit_initial) in zip(self.energy_excess_per_phase_initial, self.energy_deficit_per_phase_initial)])
        
        self.phase_groups = deque([PhaseGroup(
            group_type=PacketType.UNDEFINED,  # A phase-group of type UNDEFINED will need to be balanced first and will then be either EXCESS, DEFICIT, or BALANCED
            index_start=index_phase,
            index_end=index_phase,
            phase_pairs=self.phase_pairs
        ) for index_phase in range(self.N_phases)])
                

energy_excess_per_phase_initial = [3,2,2,3,2,6,2,4,5,7,2,2,2]
energy_deficit_per_phase_initial= [3,2,3,4,2,6,2,3,8,3,3,1,1]


#energy_per_phase_initial: List[Dict[PacketType, float]] = [{
#    PacketType.EXCESS: energy_excess_initial,
#    PacketType.DEFICIT: energy_deficit_initial
#} for (energy_excess_initial, energy_deficit_initial) in zip(energy_excess_per_phase_initial, energy_deficit_per_phase_initial)]


In [2]:
"""
Packets in each phase are stored in a deque.
Whenever a packet is balanced we rotate the deque to the left by 1.
This keeps packets in order and unbalanced packets are always at the start of the deque.
"""
    

def balance(ctx):
    print(f'Balancing: {ctx}')
    #i = ctx.indices_to_balance[0]  # will point to the first EXCESS phase
    it = 0
    lock_i = False    
    #j = None  # will point to the last EXCESS phase
    lock_j = False
    
    #k = i  # will point to the first DEFICIT phase
    lock_k = False
    #o = None  # will point to the last DEFICIT phase
    lock_o = False
    
    for phase_group in ctx.phase_groups:
        phase_group.balance_group()

        has_excess = phase_group.group_type == PacketType.EXCESS
        has_deficit = phase_group.group_type == PacketType.DEFICIT
        if has_excess:
            lock_o = False
            
        if not lock_i and has_excess:
            #i = index_phase
            ctx.indices_first_and_last[PacketType.EXCESS][0] = it
            lock_i = True

        if not lock_j and has_excess:
            #j = index_phase
            ctx.indices_first_and_last[PacketType.EXCESS][1] = it
            lock_j = True

        if has_deficit:
            lock_j = False
            
        if not lock_k and has_deficit:
            #k = index_phase
            ctx.indices_first_and_last[PacketType.DEFICIT][0] = it
            lock_k = True

        if not lock_o and has_deficit:
            #o = index_phase
            ctx.indices_first_and_last[PacketType.DEFICIT][1] = it
            lock_o = True
        
        it += 1

        
    print(f'{ctx.indices_first_and_last = }')
    return ctx.n_unbalanced_excess == 0 or ctx.n_unbalanced_deficit == 0

    

In [22]:
def merge_groups(ctx):
    if ctx.indices_first_and_last[PacketType.EXCESS][0] < ctx.indices_first_and_last[PacketType.DEFICIT][0] < ctx.indices_first_and_last[PacketType.EXCESS][1]:
        # rotate right to the last excess
        ctx.phase_groups.rotate(ctx.N_phases - ctx.indices_first_and_last[PacketType.EXCESS][1])
    else:
        # rotate left to the first excess
        ctx.phase_groups.rotate(-ctx.indices_first_and_last[PacketType.EXCESS][0])

    group_type_prev_phase = PacketType.DEFICIT

    #def create_new_temp_balance_group():
    #    return PhaseGroup(
    #        group_type = PacketType.BALANCED,
    #        index_start = None,
    #        index_end = None,
    #        capacities_for_shift = [0],
    #    )
    #
    #balance_group_tmp = create_new_temp_balance_group()
    tmp_index_start = None
    tmp_index_end = None
    tmp_capacity_for_shift = 0
    
    #def start_new_phase_group(i: int, group_type: PacketType, index_target=None):
    #    index_target = index_target if group_type != PacketType.DEFICIT else i
    #    phase_groups.append(PhaseGroup(
    #        group_type=group_type, 
    #        index_start=i, 
    #        index_target=index_target
    #    ))

    
    
    n_rotations_total = len(ctx.phase_groups)
    n_rotated = 0
    while n_rotated < n_rotations_total:
        rotate = True
        group_type_current_phase = ctx.phase_groups[0].group_type
        index_group_start = ctx.phase_groups[0].index_start
        index_group_end = ctx.phase_groups[0].index_end
        
        #if group_type_prev_phase == PacketType.DEFICIT and group_type_current_phase == PacketType.BALANCED:
        #    # at this point, we need to store the information in a temporary BALANCED group and decide on further actions, once the group type changes again
        #    tmp_capacity_for_shift = ctx.phase_groups[0].phase_pairs[index_group_start].energy_packets[PacketType.EXCESS][-1].capacity_max
    
        if group_type_prev_phase == PacketType.BALANCED and group_type_current_phase == PacketType.BALANCED:
            # merge the balance group into the previous one and update the maximum of the BALANCE group
            ctx.phase_groups[-1].index_end = ctx.phase_groups[0].index_end  # expand the group to cover both
            ctx.phase_groups[-1].capacities_for_shift[-1] = max(ctx.phase_groups[-1].capacities_for_shift[-1], ctx.phase_groups[0].capacities_for_shift[-1])
           
            ctx.phase_groups.popleft()
            n_rotations_total -= 1
            rotate = False
    
        if group_type_prev_phase == PacketType.BALANCED and group_type_current_phase == PacketType.EXCESS:
            # in this case, we need to remmeber this group as an actual BALANCE group, since no packet will be shifted past it.
            #balance_group_tmp.index_end = (i - 1) % ctx.N_Phases
            #shift_groups.append(balance_group_tmp)
            #balance_group_tmp = create_new_temp_balance_group()
            pass # nothing to do for now
    
        
        if group_type_prev_phase == PacketType.DEFICIT and group_type_current_phase == PacketType.EXCESS:
            #shift_groups[-1].index_end = (i - 1) % ctx.N_Phases
            
            # We need to start a new group!
            #start_new_phase_group( i, PacketType.EXCESS )
            pass # nothing to do for now
            
        if group_type_prev_phase == PacketType.EXCESS and group_type_current_phase == PacketType.DEFICIT:
            # set the end and target for the previous group
            ctx.phase_groups[-1].index_end = (index_group_start - 1) % ctx.N_Phases
            ctx.shift_groups[-1].index_target = index_group_start
            #shift_groups[-1].index_end = (i - 1) % ctx.N_Phases
                     
            #start_new_phase_group( i, PacketType.DEFICIT )
        
        if group_type_prev_phase == PacketType.EXCESS and group_type_current_phase == PacketType.BALANCED:
            # We can stay inside the same excess group, but need to carry along the maximum capacity of all consecutive balanced phases!
            ctx.shift_groups[-1].index_end = ctx.phase_groups[0].index_end  # expand the group to cover both
            ctx.shift_groups[-1].capacities_for_shift[-1] = max(ctx.phase_groups[0].capacities_for_shift[-1], ctx.shift_groups[-1].capacities_for_shift[-1])            
            group_type_prev_phase = PacketType.EXCESS # since we conitinue the EXCESS group, we force the group type to EXCESS and can continue.
            ctx.phase_groups.popleft()
            n_rotations_total -= 1
            rotate = False

            
        
        if group_type_current_phase == PacketType.EXCESS:
            # add current index to the shift group
            shift_groups[-1].indices_to_shift.append(i)
           
            capacity_shift = ctx.capacity_packets[i][PacketType.EXCESS][-1] + ctx.energy_packets[i][PacketType.EXCESS][-1]     
            shift_groups[-1].capacities_for_shift.append(capacity_shift)
        elif group_type_prev_phase == PacketType.DEFICIT and group_type_current_phase == PacketType.DEFICIT:
            # add current index to the shift group
            shift_groups[-1].indices_to_shift.append(i)
            
            capacity_shift = ctx.capacity_packets[i][PacketType.DEFICIT][-1] + ctx.energy_packets[i][PacketType.DEFICIT][-1]     
            shift_groups[-1].capacities_for_shift.append(capacity_shift)
        elif group_type_prev_phase == PacketType.BALANCED and group_type_current_phase == PacketType.DEFICIT:
            # in this case, we need to utilize the capacity of the BALANCE-Group for the current phase
            # add current index to the shift group
            shift_groups[-1].indices_to_shift.append(i)
            
            capacity_shift = ctx.capacity_packets[i][PacketType.DEFICIT][-1] + ctx.energy_packets[i][PacketType.DEFICIT][-1]   
            capacity_shift = max(balance_group_tmp.capacities_for_shift[-1], capacity_shift)
            shift_groups[-1].capacities_for_shift.append(capacity_shift)
            
        # set previous phase to current phase type
        group_type_prev_phase = group_type_current_phase

        if rotate:
            ctx.phase_groups.rotate(-1)
            n_rotated += 1

    
    return
    
    for i in range(i_start, i_start + N_phases): # remmeber to do modulo N_phases or exchange with rotation
        # Je index führen wir folgende Schritte durch.
        
        group_type_current_phase = PacketType.BALANCED
        if n_unbalanced[i][PacketType.EXCESS]:
            group_type_current_phase = PacketType.EXCESS
        elif n_unbalanced[i][PacketType.DEFICIT]:
            group_type_current_phase = PacketType.DEFICIT
    
        
        if group_type_prev_phase == PacketType.DEFICIT and group_type_current_phase == PacketType.BALANCED:
            # at this point, we need to store the information in a temporary BALANCED group and decide on further actions, once the group type changes again
            balance_group_tmp.index_start = i
            capacity_shift = ctx.capacity_packets[i][PacketType.EXCESS][-1] + ctx.energy_packets[i][PacketType.EXCESS][-1]
            balance_group_tmp.capacities_for_shift[-1] = capacity_shift
    
        if group_type_prev_phase == PacketType.BALANCED and group_type_current_phase == PacketType.BALANCED:
            # only update the maximum of the BALANCE group
            capacity_shift = ctx.capacity_packets[i][PacketType.EXCESS][-1] + ctx.energy_packets[i][PacketType.EXCESS][-1]
            if capacity_shift > balance_group_tmp.capacities_for_shift[-1]:
                balance_group_tmp.capacities_for_shift[-1] = capacity_shift
    
        if group_type_prev_phase == PacketType.BALANCED and group_type_current_phase == PacketType.EXCESS:
            # in this case, we need to remmeber this group as an actual BALANCE group, since no packet will be shifted past it.        
            balance_group_tmp.index_end = (i - 1) % ctx.N_Phases
            shift_groups.append(balance_group_tmp)
            balance_group_tmp = create_new_temp_balance_group()
    
        
        if group_type_prev_phase == PacketType.DEFICIT and group_type_current_phase == PacketType.EXCESS:
            shift_groups[-1].index_end = (i - 1) % ctx.N_Phases
            
            # We need to start a new group!
            start_new_phase_group( i, PacketType.EXCESS )
            
        if group_type_prev_phase == PacketType.EXCESS and group_type_current_phase == PacketType.DEFICIT:
            # set the end and target for the previous group
            shift_groups[-1].index_end = (i - 1) % ctx.N_Phases
            shift_groups[-1].index_target = i
            
            start_new_phase_group( i, PacketType.DEFICIT )
        
        if group_type_prev_phase == PacketType.EXCESS and group_type_current_phase == PacketType.BALANCED:
            # We can stay inside the same excess group, but need to carry along the maximum capacity of all consecutive balanced phases!
            capacity_shift = ctx.capacity_packets[i][PacketType.EXCESS][-1] + ctx.energy_packets[i][PacketType.EXCESS][-1]
            if capacity_shift > shift_groups[-1].capacities_for_shift[-1]:
                shift_groups[-1].capacities_for_shift[-1] = capacity_shift
                
            group_type_prev_phase = PacketType.EXCESS # since we conitinue the EXCESS group, we force the group type to EXCESS and can continue.
        
        if group_type_current_phase == PacketType.EXCESS:
            # add current index to the shift group
            shift_groups[-1].indices_to_shift.append(i)
           
            capacity_shift = ctx.capacity_packets[i][PacketType.EXCESS][-1] + ctx.energy_packets[i][PacketType.EXCESS][-1]     
            shift_groups[-1].capacities_for_shift.append(capacity_shift)
        elif group_type_prev_phase == PacketType.DEFICIT and group_type_current_phase == PacketType.DEFICIT:
            # add current index to the shift group
            shift_groups[-1].indices_to_shift.append(i)
            
            capacity_shift = ctx.capacity_packets[i][PacketType.DEFICIT][-1] + ctx.energy_packets[i][PacketType.DEFICIT][-1]     
            shift_groups[-1].capacities_for_shift.append(capacity_shift)
        elif group_type_prev_phase == PacketType.BALANCED and group_type_current_phase == PacketType.DEFICIT:
            # in this case, we need to utilize the capacity of the BALANCE-Group for the current phase
            # add current index to the shift group
            shift_groups[-1].indices_to_shift.append(i)
            
            capacity_shift = ctx.capacity_packets[i][PacketType.DEFICIT][-1] + ctx.energy_packets[i][PacketType.DEFICIT][-1]   
            capacity_shift = max(balance_group_tmp.capacities_for_shift[-1], capacity_shift)
            shift_groups[-1].capacities_for_shift.append(capacity_shift)
            
        # set previous phase to current phase type
        group_type_prev_phase = group_type_current_phase

In [ ]:
def group(ctx: Context):
    print(f'Grouping: {ctx}')
    def print_group_context(ctx):
        print('Grouping context:')
        print(f'{ctx.shift_targets = }')
        print(f'{ctx.shift_capacities = }')
        print(f'{ctx.shift_indices = }')
        print('\n')


    n_rotate_left = len(ctx.indices_to_balance)
    n_rotated = 0
    
    shift_indices_undefined = deque()
    shift_capacity = {
        PacketType.EXCESS: 0, 
        PacketType.DEFICIT: 0,
        PacketType.UNDEFINED: 0
    }
    """
    Idea:
    To remove the balance index efficiently we rotate the indices_to_balance deque and can simply popleft in the index when needed.
    This removal will reduce the total required rotation count by 1.
    Additionally we try to skip balanced phase-pairs. 
    We always have an group_type, that starts with "UNDEFINED"
    There are 3 different cases for balanced groups:
        1. We do not have any previous group yet (group_type == UNDEFINED). In that case we consider it to belong to the next group that is excess or deficit. We store it in a temporary group of type UNDEFINED and assign it later, when the group type is known.
        2. We do have a previous excess group (group_type == EXCESS). In that case it belongs to the same excess group. Any later excess group will also be added to same group. We store it in the same group and treat it as excess and might increase the shift-capacity.
        3. We do have a previous deficit group. In that case it belongs to the same deficit group. Any later deficit group will also be added to the same group. We store it in the same group and treat it as deficit and might increase the shift-capacity.
       
    
    """
    prev_group_type = PacketType.UNDEFINED
    while n_rotated < n_rotate_left:
        # check the first phase-pair
        i = ctx.indices_to_balance[0]
        current_phase_has_excess = ctx.n_unbalanced[0][PacketType.EXCESS] > 0  # The current phase-pair has unbalanced excess.
        current_phase_has_deficit = ctx.n_unbalanced[0][PacketType.DEFICIT] > 0  # The current phase-pair has unbalanced deficit.
        
        current_phase_group = PacketType.UNDEFINED
        if current_phase_has_excess:
            current_phase_group = PacketType.EXCESS
        elif current_phase_has_deficit:
            current_phase_group = PacketType.DEFICIT
    
    
    """
    Version 0
    """
    
    prev_group_type = PacketType.UNDEFINED
    ctx.shift_targets = deque()
    ctx.shift_capacities = deque([{PacketType.EXCESS: 0, PacketType.DEFICIT: 0}])
    ctx.shift_indices = deque([{PacketType.EXCESS: deque(), PacketType.DEFICIT: deque()}])
    indices_to_remove_from_balance = []
    print_group_context(ctx)

    for i in ctx.indices_to_balance:
        previous_phase_in_excess_group = prev_group_type == PacketType.EXCESS
        previous_phase_in_deficit_group = prev_group_type == PacketType.DEFICIT

        current_phase_has_excess = ctx.n_unbalanced[i][PacketType.EXCESS] > 0  # The current phase-pair has unbalanced excess.
        current_phase_has_deficit = ctx.n_unbalanced[i][PacketType.DEFICIT] > 0  # The current phase-pair has unbalanced deficit.
        
        if current_phase_has_excess and current_phase_has_deficit:  # The current phase-pair has unbalanced excess and deficit. This should be impossible!
            raise
        
        current_phase_balanced = not (current_phase_has_excess or current_phase_has_deficit) # The current phase-pair has neither unbalanced excess nor unbalanced deficit.

        previous_phase_has_excess = ctx.n_unbalanced[i-1][PacketType.EXCESS] > 0  # The previous phase-pair has unbalanced excess.
        previous_phase_has_deficit = ctx.n_unbalanced[i-1][PacketType.DEFICIT] > 0  # The previous phase-pair has unbalanced deficit.        
        previous_phase_balanced = not (previous_phase_has_excess or previous_phase_has_deficit)  # The previous phase-pair has neither unbalanced excess nor unbalanced deficit.
                
        current_phase_in_excess_group = current_phase_has_excess  # The current index belongs to an excess group. This is always the case when the current phase has excess.
        current_phase_in_excess_group |= previous_phase_in_excess_group and current_phase_balanced  # It is also the case, when we are currently constructing an excess group and are balanced.
                
        is_deficit_start = current_phase_balanced and not previous_phase_in_deficit_group  # always treat i as the shift target in balanced phases    
        is_deficit_start |= current_phase_has_deficit and previous_phase_has_excess  # also, when we change from an excess phase to a deficit phase
           
        is_excess_start = current_phase_has_excess and (previous_phase_has_deficit or previous_phase_balanced)  # If the previous phase had excess or was balanced and now we have excess.
        
        current_phase_is_shift_target = is_deficit_start

        no_edge = not is_deficit_start and not is_excess_start
        
        if no_edge and not current_phase_in_excess_group:
            current_group_type = prev_group_type
        elif is_deficit_start:
            current_group_type = PacketType.DEFICIT
        else:
            current_group_type = PacketType.EXCESS

        print(f'Index {i} in {current_group_type} group.')
        
        index_last_unbalanced_excess = ctx.n_unbalanced[i][PacketType.EXCESS] - 1
        index_last_unbalanced_deficit = ctx.n_unbalanced[i][PacketType.DEFICIT] - 1

        if current_phase_is_shift_target:  # Save index as shift-target, save minimum capacity for new packets for excess and deficit
            print(f'New shift target at {i}')
            ctx.shift_targets.append(i)

            capacity_top_of_excess_packets = ctx.capacity_packets[i][PacketType.EXCESS][index_last_unbalanced_excess] + ctx.energy_packets[i][PacketType.EXCESS][index_last_unbalanced_excess]
            capacity_top_of_deficit_packets = ctx.capacity_packets[i][PacketType.DEFICIT][index_last_unbalanced_deficit] + ctx.energy_packets[i][PacketType.DEFICIT][index_last_unbalanced_deficit]

            ctx.shift_capacities[-1][PacketType.EXCESS] = capacity_top_of_excess_packets
            ctx.shift_capacities[-1][PacketType.DEFICIT] = capacity_top_of_deficit_packets
            
            ctx.shift_capacities.append({
                PacketType.EXCESS: 0,
                PacketType.DEFICIT: 0
            })

            ctx.shift_indices.append({
                PacketType.EXCESS: deque(),
                PacketType.DEFICIT: deque()
            })  # this closes the previous excess group and prepares a new one

        else:
            indices_to_remove_from_balance.append(i)  # Any index, that is not a shift target, will never require balancing again
            if current_phase_has_excess:
                ctx.shift_indices[-1][PacketType.EXCESS].appendleft(i)  # since excess will be compressed in reverse order, we reverse the order here
            elif current_phase_has_deficit:
                ctx.shift_indices[-2][PacketType.DEFICIT].append(i)
            else:
                #capacity_top_of_excess_packets = ctx.capacity_packets[i][PacketType.EXCESS][index_last_unbalanced_excess] + ctx.energy_packets[i][PacketType.EXCESS][index_last_unbalanced_excess]            
                capacity_top_of_deficit_packets = ctx.capacity_packets[i][PacketType.DEFICIT][index_last_unbalanced_deficit] + ctx.energy_packets[i][PacketType.DEFICIT][index_last_unbalanced_deficit]
                #ctx.shift_capacities[-1][PacketType.EXCESS] = max(ctx.shift_capacities[-1][PacketType.EXCESS], capacity_top_of_excess_packets)  # store the balanced min capacity for new packets that have to move over it.
                ctx.shift_capacities[-2][PacketType.DEFICIT] = max(ctx.shift_capacities[-1][PacketType.DEFICIT], capacity_top_of_deficit_packets)  # store the balanced min capacity for new packets that have to move over it.

        print_group_context(ctx)
        prev_group_type = current_group_type

    print(indices_to_remove_from_balance)

In [25]:
ctx = Context(energy_excess_per_phase_initial=energy_excess_per_phase_initial, energy_deficit_per_phase_initial=energy_deficit_per_phase_initial)

done = balance(ctx)
if done:
    print('We are done :-)')


merge_groups(ctx)
ctx.phase_groups

Balancing: Context(energy_excess_per_phase_initial=[3, 2, 2, 3, 2, 6, 2, 4, 5, 7, 2, 2, 2], energy_deficit_per_phase_initial=[3, 2, 3, 4, 2, 6, 2, 3, 8, 3, 3, 1, 1], N_phases=13, phase_pairs=deque([PhasePair(energy_packets={<PacketType.EXCESS: 0>: deque([EnergyPacket(capacity=0, energy=3)]), <PacketType.DEFICIT: 1>: deque([EnergyPacket(capacity=0, energy=3)])}, n_unbalanced={<PacketType.EXCESS: 0>: 1, <PacketType.DEFICIT: 1>: 1}, N_unbalanced_total=2), PhasePair(energy_packets={<PacketType.EXCESS: 0>: deque([EnergyPacket(capacity=0, energy=2)]), <PacketType.DEFICIT: 1>: deque([EnergyPacket(capacity=0, energy=2)])}, n_unbalanced={<PacketType.EXCESS: 0>: 1, <PacketType.DEFICIT: 1>: 1}, N_unbalanced_total=2), PhasePair(energy_packets={<PacketType.EXCESS: 0>: deque([EnergyPacket(capacity=0, energy=2)]), <PacketType.DEFICIT: 1>: deque([EnergyPacket(capacity=0, energy=3)])}, n_unbalanced={<PacketType.EXCESS: 0>: 1, <PacketType.DEFICIT: 1>: 1}, N_unbalanced_total=2), PhasePair(energy_packets=

deque([PhaseGroup(group_type=<PacketType.EXCESS: 0>, index_start=7, index_end=7, index_target=None, indices_to_shift=[], capacities_for_shift=[]),
       PhaseGroup(group_type=<PacketType.DEFICIT: 1>, index_start=8, index_end=8, index_target=None, indices_to_shift=[], capacities_for_shift=[]),
       PhaseGroup(group_type=<PacketType.EXCESS: 0>, index_start=9, index_end=9, index_target=None, indices_to_shift=[], capacities_for_shift=[]),
       PhaseGroup(group_type=<PacketType.DEFICIT: 1>, index_start=10, index_end=10, index_target=None, indices_to_shift=[], capacities_for_shift=[]),
       PhaseGroup(group_type=<PacketType.EXCESS: 0>, index_start=11, index_end=11, index_target=None, indices_to_shift=[], capacities_for_shift=[]),
       PhaseGroup(group_type=<PacketType.EXCESS: 0>, index_start=12, index_end=12, index_target=None, indices_to_shift=[], capacities_for_shift=[]),
       PhaseGroup(group_type=<PacketType.BALANCED: 2>, index_start=0, index_end=0, index_target=None, indices_

In [ ]:

def group(
        i:int,
        packet_type:PacketType,
        previous_group_type:PacketType,
        ctx: Context
) -> (PacketType, bool):

    if ctx.n_excess_unbalanced_total == 0 or ctx.n_deficit_unbalanced_total == 0 or ctx.n_packets_total < 2:
        return PacketType.UNDEFINED, True # nothing to group. Balance index can be removed and mEfES can terminate.

    initial_step = previous_group_type == PacketType.UNDEFINED
    if initial_step:
        shift_targets = []
        shift_excess_capacities = []
        shift_deficit_capacities = []


    previous_phase_in_excess_group = previous_group_type == PacketType.EXCESS  # The previous index belongs to an excess group.
    previous_phase_in_deficit_group = previous_group_type == PacketType.DEFICIT  # The previous index belongs to a deficit group.


    if n_excess_unbalanced[i] > 0 and n_deficit_unbalanced[i] > 0:  # The current phase-pair has unbalanced excess and deficit. This should be impossible!
        raise

    current_phase_balanced = n_excess_unbalanced[i] = 0 and n_deficit_unbalanced[i] = 0  # The current phase-pair has neither unbalanced excess nor unbalanced deficit.
    current_phase_has_excess = n_excess_unbalanced[i] > 0  # The current phase-pair has unbalanced excess.
    current_phase_has_deficit = n_deficit_unbalanced[i] > 0  # The current phase-pair has unbalanced deficit.

    previous_phase_balanced = n_excess_unbalanced[i-1] = 0 and n_deficit_unbalanced[i-1] = 0  # The previous phase-pair has neither unbalanced excess nor unbalanced deficit.
    previous_phase_has_excess = n_excess_unbalanced[i-1] > 0  # The previous phase-pair has unbalanced excess.
    previous_phase_has_deficit = n_deficit_unbalanced[i-1] > 0  # The previous phase-pair has unbalanced deficit.

    current_phase_in_excess_group = current_phase_has_excess  # The current index belongs to an excess group. This is always the case when the current phase has excess.
    current_phase_in_excess_group |= previous_phase_in_excess_group and current_phase_balanced  # It is also the case, when we are currently constructing an excess group and are balanced.


    is_deficit_start = current_phase_balanced and not previous_phase_in_deficit_group  # always treat i as the shift target in balanced phases
    is_deficit_start |= current_phase_has_deficit and previous_phase_has_excess  # also, when we change from an excess phase to a deficit phase

    is_excess_start = current_phase_has_excess and (previous_phase_has_deficit or previous_phase_balanced)  # If the previous phase had excess or was balanced and now we have excess.

    current_phase_is_shift_target = is_deficit_start

    remove_from_balance = False

    no_edge = not is_deficit_start and not is_excess_start  # The previous phase-pair belongs to the same group as the current one.

    if no_edge:
        current_group_type = previous_group_type
    else if is_deficit_start:
        current_group_type = PacketType.DEFICIT
    else:
        current_group_type = PacketType.EXCESS


    if current_phase_is_shift_target:  # Save index as shift-target, save minimum capacity for new packets for excess and deficit
        shift_targets.append(i)
        shift_excess_capacities.append(capacity_excess_packets[i][-1] + energy_excess_packets[i][-1])
        shift_deficit_capacities.append(capacity_deficit_packets[i][-1] + energy_deficit_packets[i][-1])
        shift_excess_indices.append([])  # this closes the previous excess group and prepares a new one
        shift_deficit_indices.append([])  # this closes the previous deficit group and prepares a new one
    else:
        remove_from_balance = True  # Any index, that is not a shift target, will never require balancing again
        if current_phase_has_excess:
            shift_excess_indices[-1].append(i)
        elif current_phase_has_deficit:
            shift_deficit_indices[-1].append(i)
        else:
            shift_deficit_capacities[-1] = max(shift_deficit_capacities[-1] , capacity_deficit_packets[i][-1] + energy_deficit_packets[i][-1])  # store the balanced min capacity for new packets that have to move over it.


    return current_group_type, remove_from_balance